# 02 Baselines

Train logistic regression and XGBoost, tune thresholds on validation data, and evaluate test metrics.


In [ ]:
# Colab setup. Run this cell if dependencies are missing.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    # drive.mount('/content/drive')  # Optional: uncomment if storing project/data in Drive.
    !pip install -q -r requirements.txt

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('Project root:', PROJECT_ROOT)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
from src.data_download import download_dataset
from src.preprocessing import prepare_splits, fit_transform_sklearn
from src.train_baselines import train_logistic_regression, train_xgboost, predict_proba
from src.evaluate import choose_threshold_max_f1, metrics_row, save_metrics
from src.plots import plot_roc_curve, plot_precision_recall_curve, plot_confusion_matrix, plot_xgboost_feature_importance
from src.config import FIGURES_DIR, MODELS_DIR
import joblib

csv_path = download_dataset()
split = prepare_splits(csv_path)
data = fit_transform_sklearn(split)
joblib.dump(data.preprocessor, MODELS_DIR / 'sklearn_preprocessor.joblib')
print(data.X_train.shape, data.X_val.shape, data.X_test.shape)


In [ ]:
rows = []

logreg = train_logistic_regression(data.X_train, data.y_train)
logreg_val = predict_proba(logreg, data.X_val)
logreg_threshold = choose_threshold_max_f1(data.y_val, logreg_val)
logreg_test = predict_proba(logreg, data.X_test)
rows.append(metrics_row('Logistic Regression', 'test', data.y_test, logreg_test, logreg_threshold, 'val_max_f1'))
rows[-1]


In [ ]:
xgb = train_xgboost(data.X_train, data.y_train, data.X_val, data.y_val)
xgb_val = predict_proba(xgb, data.X_val)
xgb_threshold = choose_threshold_max_f1(data.y_val, xgb_val)
xgb_test = predict_proba(xgb, data.X_test)
rows.append(metrics_row('XGBoost', 'test', data.y_test, xgb_test, xgb_threshold, 'val_max_f1'))
rows[-1]


In [ ]:
metrics_df = save_metrics(rows, append=False)
metrics_df


In [ ]:
plot_roc_curve(data.y_test, logreg_test, 'Logistic Regression', FIGURES_DIR / 'logistic_regression_roc.png')
plot_precision_recall_curve(data.y_test, logreg_test, 'Logistic Regression', FIGURES_DIR / 'logistic_regression_pr.png')
plot_confusion_matrix(data.y_test, logreg_test, 'Logistic Regression', logreg_threshold, FIGURES_DIR / 'logistic_regression_confusion_matrix.png')

plot_roc_curve(data.y_test, xgb_test, 'XGBoost', FIGURES_DIR / 'xgboost_roc.png')
plot_precision_recall_curve(data.y_test, xgb_test, 'XGBoost', FIGURES_DIR / 'xgboost_pr.png')
plot_confusion_matrix(data.y_test, xgb_test, 'XGBoost', xgb_threshold, FIGURES_DIR / 'xgboost_confusion_matrix.png')
plot_xgboost_feature_importance(xgb, data.feature_names, FIGURES_DIR / 'xgboost_feature_importance.png')
